# Name That LLM — Optimized character n-gram sweep

This notebook evaluates a lightweight authorship-attribution model under prompt-grouped cross-validation.

**Fixed features:** word TF-IDF + punctuation frequencies.
**Swept feature:** raw character TF-IDF n-gram range.
**Classifier:** class-balanced LinearSVC.

The word matrix, punctuation matrix, and grouped folds are reused across configurations to reduce redundant computation.


In [ ]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ------------------------------
# Reproducible experiment config
# ------------------------------
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_PATH = Path('/kaggle/working/char_sweep_results.json')
N_SPLITS = 5
RANDOM_STATE = 42
WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGES = [(1, 5), (2, 6), (3, 7), (2, 7)]
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 1.0
MAX_ITER = 1_500

assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found: {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
required = {'text', 'model', 'prompt_id'}
missing = required.difference(train.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

print(f'Train rows: {len(train):,}')
print(f'Classes: {len(np.unique(labels))}')
print(f'Prompt groups: {len(np.unique(groups))}')
print(f'Character ranges: {CHAR_NGRAM_RANGES}')


In [ ]:
# ------------------------------
# Feature engineering utilities
# ------------------------------
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

def punctuation_matrix(items):
    """Return per-character punctuation frequencies, one row per response."""
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def make_word_vectorizer():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def make_char_vectorizer(ngram_range):
    return TfidfVectorizer(
        analyzer='char',
        ngram_range=ngram_range,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def fit_score(X_train, y_train, X_valid, y_valid):
    model = LinearSVC(
        C=SVC_C,
        class_weight='balanced',
        max_iter=MAX_ITER,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    return float(balanced_accuracy_score(y_valid, predictions))

punctuation = punctuation_matrix(texts)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
print(f'Prepared {len(folds)} fixed prompt-grouped folds.')


In [ ]:
# ------------------------------
# Claude-fable-5 data and style audit
# ------------------------------
# This is an exploratory audit, not a validation score. It checks whether the
# unusually low OOF recall may be related to data quality, formatting, length,
# duplicate responses, or broad stylistic overlap with another class.
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

AUDIT_PATH = Path('/kaggle/working/claude_fable_audit.json')

def normalized_text(text):
    return re.sub(r'\s+', ' ', text.strip().lower())

def text_stats(text):
    words = re.findall(r"\b[\w']+\b", text)
    return {
        'characters': len(text),
        'words': len(words),
        'sentences': max(len(re.findall(r'[.!?]+', text)), 0),
        'lines': text.count('\\n') + 1,
        'punctuation': sum(text.count(c) for c in PUNCTUATION),
        'digits': sum(c.isdigit() for c in text),
        'uppercase': sum(c.isupper() for c in text),
    }

stats = pd.DataFrame([text_stats(text) for text in texts])
stats['model'] = labels
stats['prompt_id'] = groups
stats['normalized_text'] = [normalized_text(text) for text in texts]

# Per-class distribution, with Claude-fable first for easy inspection.
class_stats = stats.groupby('model')[['characters','words','sentences','lines','punctuation','digits','uppercase']].agg(['count','mean','median','std']).round(4)
claude_stats = class_stats.loc['claude-fable-5'].to_dict()

# Exact normalized duplicates and cross-label conflicts.
duplicate_groups = stats.groupby('normalized_text')['model'].agg(lambda values: sorted(set(values)))
nontrivial_duplicates = duplicate_groups[duplicate_groups.index.map(len) > 0]
conflicting_duplicates = nontrivial_duplicates[nontrivial_duplicates.map(len) > 1]
claude_duplicate_rows = stats[stats['model'].eq('claude-fable-5')].groupby('normalized_text').size()

# Exploratory centroid similarity on the full data; this is not a CV score.
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3,5), min_df=2, max_features=50_000, sublinear_tf=True)
X = vectorizer.fit_transform(texts)
centroids = []
centroid_labels = []
for model_name in np.unique(labels):
    centroids.append(X[labels == model_name].mean(axis=0))
    centroid_labels.append(model_name)
centroid_matrix = np.asarray(np.vstack([np.asarray(c) for c in centroids]))
similarity = cosine_similarity(centroid_matrix)
claude_index = centroid_labels.index('claude-fable-5')
similar_models = sorted(
    [(centroid_labels[i], float(similarity[claude_index, i])) for i in range(len(centroid_labels)) if i != claude_index],
    key=lambda item: item[1], reverse=True,
)

# Prompt-group coverage check.
prompt_counts = stats.groupby(['model','prompt_id']).size().groupby(level=0).agg(['count','mean','min','max']).round(4)

summary = {
    'experiment': 'Claude-fable-5 exploratory audit',
    'rows': int(len(stats)),
    'class_counts': {str(k): int(v) for k,v in stats['model'].value_counts().items()},
    'claude_fable_stats': {str(k): float(v) for k,v in claude_stats.items()},
    'class_stats': json.loads(class_stats.to_json()),
    'exact_normalized_duplicate_groups': int(len(nontrivial_duplicates)),
    'cross_label_duplicate_groups': int(len(conflicting_duplicates)),
    'claude_fable_duplicate_rows': int(claude_duplicate_rows[claude_duplicate_rows > 1].sum()),
    'claude_fable_nearest_centroids': similar_models[:10],
    'prompt_group_coverage': json.loads(prompt_counts.to_json()),
}
AUDIT_PATH.write_text(json.dumps(summary, indent=2))
pd.DataFrame(similar_models, columns=['model','char_centroid_cosine_similarity_to_claude_fable']).to_csv('/kaggle/working/claude_fable_similarities.csv', index=False)
pd.DataFrame(class_stats).to_csv('/kaggle/working/class_style_statistics.csv')
print(json.dumps(summary, indent=2))
print('Saved Claude-fable audit artifacts to /kaggle/working/')


## Audit interpretation

This audit is exploratory. It does not select a model or claim a leaderboard improvement. It identifies data-quality and style-overlap hypotheses to test in the next grouped-CV hierarchy experiment.
